# Primeiro Ciclo - Treino sem aumentation e sem undersampling

In [ ]:
import tensorflow as tf

# Define qual dispositivo usar (GPU 0)
device_name = "/device:GPU:0"

# O bloco 'with' garante que as operações dentro dele serão alocadas na GPU.
with tf.device(device_name):
    # Crie o modelo, defina as camadas, e treine o modelo aqui.
    # Exemplo:
    # model = tf.keras.models.Sequential([...])
    # model.compile(...)
    # model.fit(train_data, epochs=...)
    pass

print(f"O modelo está sendo executado em: {device_name}")

O modelo está sendo executado em: /device:GPU:0


In [ ]:
# -*- coding: utf-8 -*-
"""
Treino normal com subset estratificado (fogo vs. não-fogo), todas as etapas:
- balanceamento opcional
- augmentations
- U-Net (VGG16 / EfficientNetB3) com BCE e Dice
- métricas streaming + sweep de threshold completo
- figuras (PDF + PNG 600 dpi) e tabelas (.csv/.tex)
"""

# =========================
# 0) HARDWARE & TF SETUP
# =========================
!nvidia-smi
!echo -n "RAM total: "; grep MemTotal /proc/meminfo

import os, sys, json, itertools, csv, random, functools, pickle
from typing import List, Tuple, Dict

import numpy as np
import pandas as pd
import tensorflow as tf

# Desliga XLA/JIT (mais estável com mixed precision no TF 2.19)
tf.config.optimizer.set_jit(False)

# Evita alocação total da GPU
for g in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except Exception:
        pass

print("TF:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

# =========================
# 1)   MODO NORMAL + SUBSAMPLE
# =========================
FAST_DEBUG = False
SWEEP_ONLY_05 = False

USE_PRETRAINED = True
FREEZE_ENCODER = False
EPOCHS = 20
BATCH_SIZE = 16          # se faltar memória, volte para 8
IMG_SIZE = (256, 256)
LR = 1e-4
PATIENCE = 10

# ---------- CONTROLES (★) ----------
# Desliga o stratified subsample da Seção 5:
USE_SUBSAMPLE = False   # (era True)

# Não aplicar undersampling na Seção 6:
APPLY_UNDERSAMPLING = False

# Não aplicar data augmentation no TREINO (Seção 7):
APPLY_TRAIN_AUGMENTATION = False
# -----------------------------------

TRAIN_MAX = 4800
VAL_MAX   = 800
TEST_MAX  = 800
TARGET_POS_FRAC = 0.30
RANDOM_SEED = 1337

# Mixed precision normal (estável com cabeça em float32 e BN sem dtype)
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("float32")
print("Policy:", mixed_precision.global_policy())
tf.keras.backend.clear_session()

# Sanity: operação pesada na GPU
with tf.device("/GPU:0"):
    a = tf.random.normal([8000, 8000])
    _ = tf.matmul(a, a)
print("OK na GPU:", _ .shape)

# =========================
# 2) VIS & SALVAMENTO
# =========================
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams["savefig.dpi"]  = 600
mpl.rcParams["figure.dpi"]   = 120
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"]  = 42

from pathlib import Path
def savefig600(fig, basepath_no_ext: str):
    try:
        fig.tight_layout()
    except Exception:
        pass
    fig.savefig(str(basepath_no_ext) + ".pdf", bbox_inches="tight", metadata={"Creator":"novo_treino.py"})
    fig.savefig(str(basepath_no_ext) + ".png", dpi=600, bbox_inches="tight", metadata={"Creator":"novo_treino.py"})

# =========================
# 3) DRIVE & CAMINHOS
# =========================
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

# >>>>>> AJUSTE SE PRECISAR (atenção ao acento composto no caminho) <<<<<<
# só uma vez por sessão:
!rsync -ah --info=progress2 "/content/drive/MyDrive/[Mestrado] Repositório da Pesquisa/data/" "/content/data_local/"

BASE_DIR = "/content/data_local"     # troque do Drive para local
# ---------------------------------------------------

FALSE_COLOR_DIR        = os.path.join(BASE_DIR, "false_color")
MASKS_DIR              = os.path.join(BASE_DIR, "masks")
IMAGE_DISTRIBUTION_DIR = os.path.join(BASE_DIR, "image_distribution")

for p in [FALSE_COLOR_DIR, MASKS_DIR, IMAGE_DISTRIBUTION_DIR]:
    if not os.path.exists(p):
        print(f"[AVISO] Diretório inexistente: {p}")

OUT_DIR = Path("/content/drive/MyDrive/segm_resultados/reports_metrics")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("OUT_DIR:", OUT_DIR)

# =========================
# 4) LISTAS & PATHS
# =========================
def read_filelist(txt_path: str) -> List[str]:
    if not os.path.isfile(txt_path):
        raise FileNotFoundError(f"Arquivo de split não encontrado: {txt_path}")
    with open(txt_path, "r", encoding="utf-8") as f:
        names = [os.path.splitext(line.strip())[0] for line in f if line.strip()]
    return names

def make_full_paths(file_list: List[str], img_dir: str, mask_dir: str) -> Tuple[List[str], List[str]]:
    image_paths = [os.path.join(img_dir,  f + ".tif") for f in file_list]
    mask_paths  = [os.path.join(mask_dir, f + ".tif") for f in file_list]
    keep_img, keep_msk = [], []
    miss = 0
    for ip, mp in zip(image_paths, mask_paths):
        if os.path.isfile(ip) and os.path.isfile(mp):
            keep_img.append(ip); keep_msk.append(mp)
        else:
            miss += 1
    if miss:
        print(f"[AVISO] {miss} pares ausentes (imagem/máscara não encontrados).")
    return keep_img, keep_msk

train_txt = os.path.join(IMAGE_DISTRIBUTION_DIR, "train.txt")
val_txt   = os.path.join(IMAGE_DISTRIBUTION_DIR, "val.txt")
test_txt  = os.path.join(IMAGE_DISTRIBUTION_DIR, "test.txt")

train_files = read_filelist(train_txt)
val_files   = read_filelist(val_txt)
train_image_paths, train_mask_paths = make_full_paths(train_files, FALSE_COLOR_DIR, MASKS_DIR)
val_image_paths,   val_mask_paths   = make_full_paths(val_files,   FALSE_COLOR_DIR, MASKS_DIR)

if os.path.isfile(test_txt):
    test_files = read_filelist(test_txt)
    test_image_paths, test_mask_paths = make_full_paths(test_files, FALSE_COLOR_DIR, MASKS_DIR)
    print(f"[INFO] test.txt encontrado: {len(test_image_paths)} amostras.")
else:
    test_image_paths, test_mask_paths = val_image_paths[:], val_mask_paths[:]
    print("[AVISO] test.txt não encontrado. Usando VAL também como TESTE.")

# =========================
# 5) SUBSAMPLE ESTRATIFICADO POR FOGO
# =========================
import imageio.v2 as imageio
random.seed(RANDOM_SEED)

def _has_fire(mask_path: str) -> bool:
    arr = imageio.imread(mask_path)
    if arr.ndim == 3:
        arr = arr[..., 0]
    arr = arr.astype(np.float32)
    thr = 0.5 * (arr.max() if arr.max() > 1 else 1.0)
    return bool((arr >= thr).any())

def stratified_subsample(img_paths, msk_paths, max_n, target_pos_frac=0.3):
    if (not USE_SUBSAMPLE) or (max_n is None) or (max_n <= 0) or (len(img_paths) <= max_n):
        return img_paths, msk_paths

    pos, neg = [], []
    for ip, mp in zip(img_paths, msk_paths):
        (pos if _has_fire(mp) else neg).append((ip, mp))

    n_pos_avail, n_neg_avail = len(pos), len(neg)
    n_pos_target = int(round(max_n * float(target_pos_frac)))
    n_neg_target = max_n - n_pos_target

    n_pos_take = min(n_pos_target, n_pos_avail)
    n_neg_take = min(n_neg_target + (n_pos_target - n_pos_take), n_neg_avail)

    while (n_pos_take + n_neg_take) < max_n:
        if n_neg_take < n_neg_avail:
            n_neg_take += 1
        elif n_pos_take < n_pos_avail:
            n_pos_take += 1
        else:
            break

    random.shuffle(pos); random.shuffle(neg)
    picked = pos[:n_pos_take] + neg[:n_neg_take]
    random.shuffle(picked)

    new_img = [p[0] for p in picked]
    new_msk = [p[1] for p in picked]
    print(f"[SUBSAMPLE] max_n={max_n} | pos_avail={n_pos_avail} neg_avail={n_neg_avail} "
          f"=> take_pos={n_pos_take} take_neg={n_neg_take} -> total={len(new_img)}")
    return new_img, new_msk

if USE_SUBSAMPLE:
    train_image_paths, train_mask_paths = stratified_subsample(
        train_image_paths, train_mask_paths, TRAIN_MAX, TARGET_POS_FRAC
    )
    val_image_paths, val_mask_paths = stratified_subsample(
        val_image_paths, val_mask_paths, VAL_MAX, TARGET_POS_FRAC
    )
    if 'test_image_paths' in locals() and len(test_image_paths) > 0:
        test_image_paths, test_mask_paths = stratified_subsample(
            test_image_paths, test_mask_paths, TEST_MAX, TARGET_POS_FRAC
        )

def count_fire_pairs(msk_paths):
    return sum(_has_fire(m) for m in msk_paths)

print(f"[CHECK] TREINO: {len(train_mask_paths)} amostras | com fogo: {count_fire_pairs(train_mask_paths)}")
print(f"[CHECK] VAL   : {len(val_mask_paths)} amostras | com fogo: {count_fire_pairs(val_mask_paths)}")
print(f"[CHECK] TESTE : {len(test_mask_paths)} amostras | com fogo: {count_fire_pairs(test_mask_paths)}")

# =========================
# 6) BALANCEAMENTO (opcional)
# =========================
from PIL import Image
from tqdm import tqdm

def balancear_dataset_por_undersampling(image_paths: List[str], mask_paths: List[str]):
    positivos_img, positivos_mask, negativos_img, negativos_mask = [], [], [], []
    print("[INFO] Analisando máscaras para balanceamento...")
    for img_p, mask_p in tqdm(list(zip(image_paths, mask_paths)), total=len(image_paths)):
        if not os.path.exists(mask_p):
            continue
        with Image.open(mask_p) as mask:
            arr = np.array(mask)
            if np.sum(arr) > 0:
                positivos_img.append(img_p); positivos_mask.append(mask_p)
            else:
                negativos_img.append(img_p); negativos_mask.append(mask_p)
    num_pos, num_neg = len(positivos_img), len(negativos_img)
    print(f"[INFO] Positivas: {num_pos} | Negativas: {num_neg}")

    if num_pos == 0 or num_neg == 0:
        print("[WARN] Desbalanceado extremo. Retornando original.")
        combined = list(zip(image_paths, mask_paths)); random.shuffle(combined)
        new_image_paths, new_mask_paths = zip(*combined)
        return list(new_image_paths), list(new_mask_paths)

    if num_pos > num_neg:
        reduzir = num_neg
        print(f"[INFO] Reduzindo positivas de {num_pos} -> {reduzir}")
        pos_sample = random.sample(list(zip(positivos_img, positivos_mask)), reduzir)
        final_pos_img, final_pos_mask = zip(*pos_sample)
        final_neg_img, final_neg_mask = negativos_img, negativos_mask
    elif num_neg > num_pos:
        reduzir = num_pos
        print(f"[INFO] Reduzindo negativas de {num_neg} -> {reduzir}")
        neg_sample = random.sample(list(zip(negativos_img, negativos_mask)), reduzir)
        final_neg_img, final_neg_mask = zip(*neg_sample)
        final_pos_img, final_pos_mask = positivos_img, positivos_mask
    else:
        print("[INFO] Já balanceado.")
        final_pos_img, final_pos_mask = positivos_img, positivos_mask
        final_neg_img, final_neg_mask = negativos_img, negativos_mask

    new_image_paths = list(final_pos_img) + list(final_neg_img)
    new_mask_paths  = list(final_pos_mask) + list(final_neg_mask)
    combined = list(zip(new_image_paths, new_mask_paths)); random.shuffle(combined)
    new_image_paths, new_mask_paths = zip(*combined)
    print(f"[INFO] Balanceamento concluído. Total final: {len(new_image_paths)}")
    return list(new_image_paths), list(new_mask_paths)

# ----------- AQUI (★): respeita a flag APPLY_UNDERSAMPLING ----------
if APPLY_UNDERSAMPLING:
    train_image_paths_bal, train_mask_paths_bal = balancear_dataset_por_undersampling(
        train_image_paths, train_mask_paths
    )
else:
    print("[INFO] Undersampling desativado: usando dataset de treino ORIGINAL.")
    train_image_paths_bal, train_mask_paths_bal = train_image_paths, train_mask_paths
# --------------------------------------------------------------------

# =========================
# 7) AUGMENTATIONS
# =========================
import albumentations as A

def get_train_augmentations(img_size: Tuple[int, int]):
    return A.Compose([
        A.Resize(height=img_size[0], width=img_size[1]),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.RandomBrightnessContrast(p=0.2),
    ])

def get_val_augmentations(img_size: Tuple[int, int]):
    return A.Compose([A.Resize(height=img_size[0], width=img_size[1])])

# ----------- AQUI (★): sem augmentation no treino quando desligado ----------
if APPLY_TRAIN_AUGMENTATION:
    train_augs = get_train_augmentations(IMG_SIZE)
else:
    print("[INFO] Data augmentation no TREINO desativada: aplicando apenas Resize.")
    train_augs = get_val_augmentations(IMG_SIZE)
# Val/test sempre apenas resize
val_augs   = get_val_augmentations(IMG_SIZE)
# ---------------------------------------------------------------------------

# =========================
# 8) LOADER .tif + tf.data
# =========================
import imageio.v2 as imageio

def _read_tiff_rgb(img_path: str) -> np.ndarray:
    arr = imageio.imread(img_path)
    if arr.ndim == 2:
        arr = np.stack([arr, arr, arr], axis=-1)
    elif arr.ndim == 3:
        if arr.shape[-1] == 4:   arr = arr[..., :3]
        elif arr.shape[-1] > 3:  arr = arr[..., :3]
        elif arr.shape[-1] == 1: arr = np.repeat(arr, 3, axis=-1)
    arr = arr.astype(np.float32)
    maxv = arr.max() if arr.size else 1.0
    if maxv > 1.0:
        arr = arr / (255.0 if maxv <= 255 else maxv)
    return arr

def _read_tiff_mask(msk_path: str) -> np.ndarray:
    arr = imageio.imread(msk_path)
    if arr.ndim == 3:
        arr = arr[..., 0]
    arr = arr.astype(np.float32)
    thr = 0.5 * (arr.max() if arr.max() > 1 else 1.0)
    binm = (arr >= thr).astype(np.float32)
    binm = np.expand_dims(binm, axis=-1)
    return binm

def _apply_augs_np(img, msk, augs):
    if augs is None:
        return img, msk
    try:
        out = augs(image=img, mask=msk.squeeze(-1))
        img2, msk2 = out["image"], out["mask"]
        if msk2.ndim == 2:
            msk2 = np.expand_dims(msk2, axis=-1)
        msk2 = (msk2 >= 0.5).astype(np.float32)
        return img2.astype(np.float32), msk2.astype(np.float32)
    except Exception:
        return img, msk

def _resize_np(img: np.ndarray, msk: np.ndarray, size=IMG_SIZE):
    try:
        import cv2
        img_r = cv2.resize(img, dsize=(size[1], size[0]), interpolation=cv2.INTER_LINEAR)
        msk_r = cv2.resize(msk, dsize=(size[1], size[0]), interpolation=cv2.INTER_NEAREST)
    except Exception:
        img_r = tf.image.resize(img, size, method="bilinear").numpy()
        msk_r = tf.image.resize(msk, size, method="nearest").numpy()
    if msk_r.ndim == 2:
        msk_r = msk_r[..., None]
    return img_r.astype(np.float32), msk_r.astype(np.float32)

def load_pair_np(img_path_b, msk_path_b, size=IMG_SIZE, augs=None):
    ip = img_path_b.decode("utf-8"); mp = msk_path_b.decode("utf-8")
    img = _read_tiff_rgb(ip)
    msk = _read_tiff_mask(mp)
    img, msk = _resize_np(img, msk, size=size)
    img, msk = _apply_augs_np(img, msk, augs)
    if msk.ndim == 2: msk = msk[..., None]
    msk = (msk >= 0.5).astype(np.float32)
    img = np.clip(img, 0.0, 1.0).astype(np.float32)
    return img, msk

def _py_load(img_p, msk_p, size, augs):
    img, msk = tf.numpy_function(
        func=lambda ip, mp: load_pair_np(ip, mp, size=size, augs=augs),
        inp=[img_p, msk_p],
        Tout=[tf.float32, tf.float32]
    )
    img.set_shape((IMG_SIZE[0], IMG_SIZE[1], 3))
    msk.set_shape((IMG_SIZE[0], IMG_SIZE[1], 1))
    return img, msk

def make_tfds(image_paths: List[str], mask_paths: List[str], batch_size: int,
              shuffle: bool, augs=None, cache=True):
    ds = tf.data.Dataset.from_tensor_slices((image_paths, mask_paths))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(image_paths), 2000), reshuffle_each_iteration=True)
    ds = ds.map(lambda ip, mp: _py_load(ip, mp, IMG_SIZE, augs), num_parallel_calls=tf.data.AUTOTUNE)
    if cache:
        ds = ds.cache()
    ds = ds.batch(batch_size, drop_remainder=False).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_tfds(train_image_paths_bal, train_mask_paths_bal, BATCH_SIZE, True,  augs=train_augs, cache=True)
val_ds   = make_tfds(val_image_paths,       val_mask_paths,       BATCH_SIZE, False, augs=val_augs,   cache=True)
test_ds  = make_tfds(test_image_paths,      test_mask_paths,      BATCH_SIZE, False, augs=None,       cache=True)

for xb, yb in train_ds.take(1):
    print("[OK] Batch exemplo:", xb.shape, yb.shape)

# =========================
# 9) MÉTRICAS STREAMING
# =========================
class StreamingConfusion(tf.keras.metrics.Metric):
    def __init__(self, threshold=0.5, name="streaming_conf", **kwargs):
        super().__init__(name=name, **kwargs)
        self.threshold = threshold
        self.tp = self.add_weight(name="tp", shape=(), initializer="zeros", dtype=tf.float32)
        self.fp = self.add_weight(name="fp", shape=(), initializer="zeros", dtype=tf.float32)
        self.fn = self.add_weight(name="fn", shape=(), initializer="zeros", dtype=tf.float32)
        self.tn = self.add_weight(name="tn", shape=(), initializer="zeros", dtype=tf.float32)
    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = tf.cast(y_true > 0.5, tf.float32)
        y_pred = tf.cast(y_pred >= self.threshold, tf.float32)
        tp = tf.reduce_sum(y_true * y_pred)
        fp = tf.reduce_sum((1.0 - y_true) * y_pred)
        fn = tf.reduce_sum(y_true * (1.0 - y_pred))
        tn = tf.reduce_sum((1.0 - y_true) * (1.0 - y_pred))
        self.tp.assign_add(tp); self.fp.assign_add(fp)
        self.fn.assign_add(fn); self.tn.assign_add(tn)
    def reset_state(self):
        for v in (self.tp, self.fp, self.fn, self.tn):
            v.assign(0.0)

class StreamingPrecision(StreamingConfusion):
    def result(self):
        denom = self.tp + self.fp
        return tf.where(denom > 0, self.tp / denom, 0.0)

class StreamingRecall(StreamingConfusion):
    def result(self):
        denom = self.tp + self.fn
        return tf.where(denom > 0, self.tp / denom, 0.0)

class StreamingF1(StreamingConfusion):
    def result(self):
        num = 2.0 * self.tp
        denom = 2.0 * self.tp + self.fp + self.fn
        return tf.where(denom > 0, num / denom, 0.0)

class StreamingIoU(StreamingConfusion):
    def result(self):
        denom = self.tp + self.fp + self.fn
        return tf.where(denom > 0, self.tp / denom, 0.0)

# =========================
# 10) LOSSES & COMPILAÇÃO
# =========================
def dice_loss(y_true, y_pred, smooth: float = 1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1,2,3])
    denom = tf.reduce_sum(y_true + y_pred, axis=[1,2,3])
    dice = (2.0 * intersection + smooth) / (denom + smooth)
    return 1.0 - dice

BCE_LOSS = tf.keras.losses.BinaryCrossentropy(from_logits=False)

LOSS_FUNCS = {
    "BCE":  BCE_LOSS,
    "DICE": dice_loss,
    # "BCE_DICE": lambda y_true, y_pred: 0.5*BCE_LOSS(y_true, y_pred) + 0.5*dice_loss(y_true, y_pred),
}

def compile_with_streaming(model, loss_fn, threshold=0.5, lr=LR):
    opt = tf.keras.optimizers.Adam(learning_rate=lr)
    metrics = [
        StreamingIoU(threshold=threshold, name="iou"),
        StreamingF1 (threshold=threshold, name="f1"),
        StreamingPrecision(threshold=threshold, name="precision"),
        StreamingRecall  (threshold=threshold, name="recall"),
    ]
    model.compile(optimizer=opt, loss=loss_fn, metrics=metrics)
    return model

# =========================
# 11) MODELO (U-NET)
# =========================
from tensorflow.keras import layers as L
from tensorflow.keras.applications import VGG16, EfficientNetB3

def _bn32(x):
    return L.BatchNormalization(dtype="float32")(x)

def conv_block(x, filters):
    x = L.Conv2D(filters, 3, padding="same")(x); x = _bn32(x); x = L.ReLU()(x)
    x = L.Conv2D(filters, 3, padding="same")(x); x = _bn32(x); x = L.ReLU()(x)
    return x

def up_block(x, skip, filters):
    x = L.UpSampling2D((2,2), interpolation="bilinear")(x)
    x = tf.keras.layers.Resizing(skip.shape[1], skip.shape[2], interpolation="bilinear")(x)
    x = L.Concatenate()([x, skip])
    x = conv_block(x, filters)
    return x

def _check_downsample_ratio(inp_shape_hw, feat_shape_hw, expected_ratio):
    H_in, W_in = int(inp_shape_hw[0]), int(inp_shape_hw[1])
    H_f,  W_f  = int(feat_shape_hw[0]), int(feat_shape_hw[1])
    assert H_f > 0 and W_f > 0, "Shape dinâmico demais; forneça IMG_SIZE fixo."
    assert H_in % H_f == 0 and W_in % W_f == 0, "Entrada não é múltipla do mapa de feature."
    rh, rw = H_in // H_f, W_in // W_f
    assert rh == expected_ratio and rw == expected_ratio, (
        f"Esperado downsample {expected_ratio}×, obtido {rh}×{rw} (H×W)."
    )

def _vgg16_endpoints(base, inputs_shape):
    s1 = base.get_layer("block1_conv2").output   # 1x
    s2 = base.get_layer("block2_conv2").output   # 2x
    s3 = base.get_layer("block3_conv3").output   # 4x
    s4 = base.get_layer("block4_conv3").output   # 8x
    b  = base.get_layer("block5_conv3").output   # 16x
    _check_downsample_ratio(inputs_shape[:2], s1.shape[1:3], 1)
    _check_downsample_ratio(inputs_shape[:2], s2.shape[1:3], 2)
    _check_downsample_ratio(inputs_shape[:2], s3.shape[1:3], 4)
    _check_downsample_ratio(inputs_shape[:2], s4.shape[1:3], 8)
    _check_downsample_ratio(inputs_shape[:2], b.shape[1:3], 16)
    return s1, s2, s3, s4, b

def _efficientnetb3_endpoints(base, inputs_shape):
    # Alvos típicos p/ 256x256 (confira no print):
    # block2b_add -> 64x64  (/4)
    # block3b_add -> 32x32  (/8)
    # block4c_add -> 16x16  (/16)
    # block5c_add -> 16x16  (/16)
    # block6d_add -> 8x8    (/32)  (bottleneck)
    s0 = base.get_layer("block2b_add").output
    s1 = base.get_layer("block3b_add").output
    s2 = base.get_layer("block4c_add").output
    s3 = base.get_layer("block5c_add").output
    b  = base.get_layer("block6d_add").output
    # checagens suaves (não falha se algum stride muda entre versões)
    return s0, s1, s2, s3, b

def build_unet(backbone_name="vgg16", input_shape=(256,256,3), encoder_trainable=False):
    inputs = L.Input(shape=input_shape)

    if backbone_name.lower() == "vgg16":
        base = VGG16(include_top=False, weights="imagenet", input_tensor=inputs)
        for l in base.layers: l.trainable = encoder_trainable
        s1, s2, s3, s4, b = _vgg16_endpoints(base, input_shape)
        x = up_block(b,  s4, 512)
        x = up_block(x, s3, 256)
        x = up_block(x, s2, 128)
        x = up_block(x, s1, 64)

    elif backbone_name.lower() == "efficientnetb3":
        base = EfficientNetB3(include_top=False, weights="imagenet", input_tensor=inputs)
        for l in base.layers: l.trainable = encoder_trainable
        s0, s1, s2, s3, b = _efficientnetb3_endpoints(base, input_shape)
        x = up_block(b,  s3, 512)   # 1/16 -> 1/16
        x = up_block(x, s2, 256)    # 1/16 -> 1/8
        x = up_block(x, s1, 128)    # 1/8  -> 1/4
        x = up_block(x, s0, 64)     # 1/4  -> 1/2
        x = tf.keras.layers.Resizing(input_shape[0], input_shape[1], interpolation="bilinear")(x)
    else:
        raise ValueError("backbone_name deve ser 'vgg16' ou 'efficientnetb3'.")

    outputs = L.Conv2D(1, 1, activation="sigmoid", dtype="float32")(x)
    model = tf.keras.Model(inputs, outputs, name=f"UNet_{backbone_name}")

    print("\n[BACKBONE]", backbone_name.upper())
    print("Entrada :", input_shape)
    for lyr in [l for l in base.layers if l.name in {
        "block1_conv2","block2_conv2","block3_conv3","block4_conv3","block5_conv3",
        "block2b_add","block3b_add","block4c_add","block5c_add","block6d_add"
    }]:
        try:
            print(f" - {lyr.name:>14} -> {lyr.output.shape}")
        except:
            pass

    return model

# =========================
# 12) TREINO + PLOTS
# =========================
def fit_model(model, train_ds, val_ds, tag):
    ckpt_path = OUT_DIR / f"best_{tag}.weights.h5"
    cbs = [
        tf.keras.callbacks.EarlyStopping(monitor="val_f1", mode="max",
                                         patience=PATIENCE, restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint(filepath=str(ckpt_path),
                                           monitor="val_f1", mode="max",
                                           save_best_only=True, save_weights_only=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_f1", mode="max",
                                             factor=0.5, patience=max(3, PATIENCE//2), verbose=1),
    ]
    hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
                     callbacks=cbs, verbose=1)
    return hist, ckpt_path

from matplotlib.ticker import MaxNLocator, AutoMinorLocator

def _style_axes_epoch(ax, epochs_total=None):
    ax.set_axisbelow(True)
    ax.grid(True, which="major", linestyle="--", linewidth=0.8, alpha=0.35)
    ax.grid(True, which="minor", linestyle=":",  linewidth=0.6, alpha=0.25)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.xaxis.set_minor_locator(AutoMinorLocator(2))
    if epochs_total is not None and epochs_total > 0:
        ax.set_xlim(0, max(1, int(epochs_total)))
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Metric Value")
    leg = ax.legend(frameon=True); leg.get_frame().set_alpha(0.8)

def plot_history(history, title, fname_base):
    # Loss
    fig, ax = plt.subplots(figsize=(7,4), dpi=150)
    ax.plot(history.history.get("loss", []),     label="Loss (train)",   linewidth=1.5)
    ax.plot(history.history.get("val_loss", []), label="Loss (val)",     linewidth=1.5)
    ax.set_title(f"{title} — Loss")
    _style_axes_epoch(ax, epochs_total=len(history.history.get("loss", [])))
    savefig600(fig, OUT_DIR / f"{fname_base}_loss")
    plt.close(fig)

    # Métricas
    fig, ax = plt.subplots(figsize=(7,4), dpi=150)
    if "precision" in history.history:   ax.plot(history.history["precision"],   label="Precision",   linewidth=1.5)
    if "recall" in history.history:      ax.plot(history.history["recall"],      label="Recall",      linewidth=1.5)
    if "iou" in history.history:         ax.plot(history.history["iou"],         label="IoU",         linewidth=1.5)
    if "f1" in history.history:          ax.plot(history.history["f1"],          label="F1 Score",    linewidth=1.5)
    if "val_iou" in history.history:     ax.plot(history.history["val_iou"],     label="Val IoU",     linewidth=1.5)
    if "val_f1" in history.history:      ax.plot(history.history["val_f1"],      label="Val F1",      linewidth=1.5)
    if "val_precision" in history.history: ax.plot(history.history["val_precision"], label="Val Precision", linewidth=1.5)
    if "val_recall" in history.history:    ax.plot(history.history["val_recall"],    label="Val Recall",    linewidth=1.5)
    ax.set_title(f"{title} — Metrics Over Epochs")
    ax.set_ylim(0.0, 1.0)
    _style_axes_epoch(ax, epochs_total=len(history.history.get("loss", [])))
    savefig600(fig, OUT_DIR / f"{fname_base}_metrics")
    plt.close(fig)

# =========================
# 13) SWEEP, AVAL, CM, TABELA
# =========================
def sweep_threshold(model, val_ds, thresholds=None):
    if thresholds is None:
        thresholds = np.linspace(0.05, 0.95, 19) if not SWEEP_ONLY_05 else np.array([0.5])
    best_t, best_f1 = thresholds[0], -1.0
    rows = []
    for t in thresholds:
        TP=FP=FN=TN=0
        for x, y in val_ds:
            y_true = (y.numpy() > 0.5).astype(np.uint8)
            y_prob = model.predict(x, verbose=0)
            y_pred = (y_prob >= t).astype(np.uint8)
            TP += (y_true & y_pred).sum()
            FP += ((1 - y_true) & y_pred).sum()
            FN += (y_true & (1 - y_pred)).sum()
            TN += ((1 - y_true) & (1 - y_pred)).sum()
        precision = TP/(TP+FP) if (TP+FP)>0 else 0.0
        recall    = TP/(TP+FN) if (TP+FN)>0 else 0.0
        f1        = (2*precision*recall)/(precision+recall) if (precision+recall)>0 else 0.0
        iou       = TP/(TP+FP+FN) if (TP+FP+FN)>0 else 0.0
        rows.append(dict(threshold=t, precision=precision, recall=recall, f1=f1, iou=iou,
                         TP=int(TP), FP=int(FP), FN=int(FN), TN=int(TN)))
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, pd.DataFrame(rows)

def evaluate_dataset(model, dataset, threshold=0.5):
    TP=FP=FN=TN=0
    for x, y in dataset:
        y_true = (y.numpy() > 0.5).astype(np.uint8)
        y_prob = model.predict(x, verbose=0)
        y_pred = (y_prob >= threshold).astype(np.uint8)
        TP += (y_true & y_pred).sum()
        FP += ((1 - y_true) & y_pred).sum()
        FN += (y_true & (1 - y_pred)).sum()
        TN += ((1 - y_true) & (1 - y_pred)).sum()
    precision = TP/(TP+FP) if (TP+FP)>0 else 0.0
    recall    = TP/(TP+FN) if (TP+FN)>0 else 0.0
    f1        = (2*precision*recall)/(precision+recall) if (precision+recall)>0 else 0.0
    iou       = TP/(TP+FP+FN) if (TP+FP+FN)>0 else 0.0
    cm = np.array([[TN, FP],[FN, TP]], dtype=np.int64)
    return dict(precision=precision, recall=recall, f1=f1, iou=iou,
                TP=int(TP), FP=int(FP), FN=int(FN), TN=int(TN), threshold=threshold), cm

import numpy as np
import matplotlib.pyplot as plt

def plot_confusion_matrix(
    cm,
    classes=("0","1"),
    title="Confusion Matrix",
    normalize=None,
    png_path=None,
    pdf_path=None,
):
    cm = np.asarray(cm)
    show_norm = normalize in ("true", "all")
    cm_plot = cm.astype(np.float64).copy()

    if normalize == "true":
        row_sums = cm_plot.sum(axis=1, keepdims=True)
        cm_plot = np.divide(cm_plot, row_sums, out=np.zeros_like(cm_plot), where=row_sums > 0.0)
        vmin, vmax = 0.0, 1.0
    elif normalize == "all":
        tot = cm_plot.sum()
        cm_plot = cm_plot / tot if tot > 0 else cm_plot
        vmin, vmax = 0.0, 1.0
    else:
        vmin, vmax = 0, cm_plot.max() if cm_plot.size else 1

    fig, ax = plt.subplots(figsize=(6, 6), dpi=150)
    im = ax.imshow(cm_plot, cmap="Greens", vmin=vmin, vmax=vmax, interpolation="nearest")
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.set_ylabel("Proporção" if show_norm else "Contagem", rotation=270, labelpad=12)

    ax.set_title(title, pad=10)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_xticks(np.arange(len(classes)), labels=classes)
    ax.set_yticks(np.arange(len(classes)), labels=classes)
    ax.set_aspect("equal")

    thresh = (cm_plot.max() / 2.0) if cm_plot.size else 0.0
    for i in range(cm_plot.shape[0]):
        for j in range(cm_plot.shape[1]):
            if show_norm:
                text = f"{cm_plot[i, j]:.2f}\n({int(cm[i, j])})"
            else:
                text = f"{int(cm_plot[i, j])}"
            ax.text(
                j, i, text,
                ha="center", va="center",
                fontsize=12,
                color="white" if cm_plot[i, j] > thresh else "black",
            )

    plt.tight_layout()
    if pdf_path: fig.savefig(pdf_path, bbox_inches="tight")
    if png_path: fig.savefig(png_path, dpi=600, bbox_inches="tight")
    plt.close(fig)

def save_table2(val_metrics, test_metrics, backbone_tag_loss, out_dir=OUT_DIR):
    # backbone_tag_loss ex.: "vgg16_BCE" ou "efficientnetb3_DICE"
    backbone_tag, loss_name = backbone_tag_loss.split("_", 1)
    df = pd.DataFrame([
        {"Modelo": f"{backbone_tag.upper()} ({loss_name})", "Conjunto":"Validação",
         "Threshold": f"{val_metrics['threshold']:.2f}",
         "Precisão": val_metrics["precision"],
         "Recall":   val_metrics["recall"],
         "F1-Score": val_metrics["f1"],
         "IoU":      val_metrics["iou"]},
        {"Modelo": f"{backbone_tag.upper()} ({loss_name})", "Conjunto":"Teste",
         "Threshold": f"{test_metrics['threshold']:.2f}",
         "Precisão": test_metrics["precision"],
         "Recall":   test_metrics["recall"],
         "F1-Score": test_metrics["f1"],
         "IoU":      test_metrics["iou"]},
    ])
    csv_path = out_dir / f"tabela2_{backbone_tag}_{loss_name}.csv"
    tex_path = out_dir / f"tabela2_{backbone_tag}_{loss_name}.tex"
    df.to_csv(csv_path, index=False)
    with open(tex_path, "w", encoding="utf-8") as f:
        f.write(df.to_latex(index=False, float_format="%.4f", escape=False,
                caption=(f"Tabela 2 — {backbone_tag.upper()} (loss: {loss_name}): "
                         "métricas por matrizes de confusão (Validação/Teste) e limiar."),
                label=f"tab:metrics_{backbone_tag}_{loss_name}"))
    print(f"[OK] {backbone_tag}_{loss_name}: Tabela 2 salva em {csv_path} e {tex_path}")
    return df

def run_pipeline_for(backbone_tag, loss_name, train_ds, val_ds, test_ds):
    print(f"\n========== {backbone_tag.upper()} ({loss_name}) ==========")
    model = build_unet(backbone_tag, input_shape=(*IMG_SIZE, 3))
    model = compile_with_streaming(model, loss_fn=LOSS_FUNCS[loss_name], threshold=0.5, lr=LR)

    tag = f"{backbone_tag}_{loss_name}"
    history, ckpt_path = fit_model(model, train_ds, val_ds, tag)
    fig_id = "8" if backbone_tag.lower()=="vgg16" else "9"
    plot_history(history, f"U-Net ({backbone_tag.upper()} / {loss_name})",
                 fname_base=f"Figura_{fig_id}_{backbone_tag}_{loss_name}_curvas")

    model.load_weights(str(ckpt_path))

    best_t, sweep_df = sweep_threshold(model, val_ds)
    json_path = OUT_DIR / "best_thresholds.json"
    try:
        data = json.loads(json_path.read_text())
    except Exception:
        data = {}
    if loss_name not in data: data[loss_name] = {}
    data[loss_name][backbone_tag] = float(best_t)
    json_path.write_text(json.dumps(data, indent=2))

    sweep_df.to_csv(OUT_DIR / f"sweep_{backbone_tag}_{loss_name}.csv", index=False)
    print(f"[{backbone_tag}/{loss_name}] Threshold ótimo (val/F1): {best_t:.2f}")

    val_metrics,  val_cm  = evaluate_dataset(model, val_ds,  threshold=best_t)
    test_metrics, test_cm = evaluate_dataset(model, test_ds, threshold=best_t)
    print(f"[{backbone_tag}/{loss_name}] VAL:", val_metrics)
    print(f"[{backbone_tag}/{loss_name}] TEST:", test_metrics)

    # Figuras (Val/Test) — absoluta e normalizada
    plot_confusion_matrix(val_cm,  title=f"{backbone_tag.upper()} — Validação (thr={best_t:.2f})",
                          normalize=None,
                          png_path=OUT_DIR / f"Figura_10_{backbone_tag}_{loss_name}_val_cm.png",
                          pdf_path=OUT_DIR / f"Figura_10_{backbone_tag}_{loss_name}_val_cm.pdf")
    plot_confusion_matrix(val_cm,  title=f"{backbone_tag.upper()} — Validação Normalizada (thr={best_t:.2f})",
                          normalize='true',
                          png_path=OUT_DIR / f"Figura_10_{backbone_tag}_{loss_name}_val_cm_norm.png",
                          pdf_path=OUT_DIR / f"Figura_10_{backbone_tag}_{loss_name}_val_cm_norm.pdf")
    plot_confusion_matrix(test_cm, title=f"{backbone_tag.upper()} — Teste (thr={best_t:.2f})",
                          normalize=None,
                          png_path=OUT_DIR / f"Figura_11_{backbone_tag}_{loss_name}_test_cm.png",
                          pdf_path=OUT_DIR / f"Figura_11_{backbone_tag}_{loss_name}_test_cm.pdf")
    plot_confusion_matrix(test_cm, title=f"{backbone_tag.upper()} — Teste Normalizada (thr={best_t:.2f})",
                          normalize='true',
                          png_path=OUT_DIR / f"Figura_11_{backbone_tag}_{loss_name}_test_cm_norm.png",
                          pdf_path=OUT_DIR / f"Figura_11_{backbone_tag}_{loss_name}_test_cm_norm.pdf")

    _ = save_table2(val_metrics, test_metrics, f"{backbone_tag}_{loss_name}", OUT_DIR)
    return dict(backbone=backbone_tag, loss=loss_name,
                best_threshold=best_t, val=val_metrics, test=test_metrics)

# =========================
# 14) EXECUTAR
# =========================
BACKBONES = ["vgg16", "efficientnetb3"]
LOSSES_TO_RUN = ["BCE", "DICE"]  # adicione "BCE_DICE" se habilitar no LOSS_FUNCS

results = []
for loss_name in LOSSES_TO_RUN:
    for bb in BACKBONES:
        results.append(run_pipeline_for(bb, loss_name, train_ds, val_ds, test_ds))

# Tabela comparativa geral (Figura 2)
rows = []
for r in results:
    rows += [
        {"Backbone": r["backbone"].upper(), "Loss": r["loss"], "Conjunto":"Validação",
         "Threshold": f"{r['best_threshold']:.2f}",
         "Precisão": r["val"]["precision"], "Recall": r["val"]["recall"],
         "F1-Score": r["val"]["f1"], "IoU": r["val"]["iou"]},
        {"Backbone": r["backbone"].upper(), "Loss": r["loss"], "Conjunto":"Teste",
         "Threshold": f"{r['best_threshold']:.2f}",
         "Precisão": r["test"]["precision"], "Recall": r["test"]["recall"],
         "F1-Score": r["test"]["f1"], "IoU": r["test"]["iou"]},
    ]

df_all = pd.DataFrame(rows)
df_all = df_all.sort_values(["Backbone","Loss","Conjunto"])

df_all.to_csv(OUT_DIR / "Figura_2_tabela2_comparativo.csv", index=False)
with open(OUT_DIR / "Figura_2_tabela2_comparativo.tex", "w", encoding="utf-8") as f:
    f.write(df_all.to_latex(index=False, float_format="%.4f", escape=False,
            caption=("Tabela 2 — Comparativo (Backbones × Funções de Perda) "
                     "com métricas pixel-a-pixel calculadas a partir das matrizes de confusão, "
                     "no limiar ótimo de validação."),
            label="tab:tabela2_comparativo"))

print("\n[PRONTO] Artefatos em:", OUT_DIR.resolve())
print(" - Figuras (curvas e CMs) em PDF + PNG 600dpi")
print(" - Tabelas .csv e .tex (Val/Teste + threshold) por backbone+loss")
print(" - Figura_2_tabela2_comparativo.(csv|tex)")
print(f" - Subsample (train/val/test max): {TRAIN_MAX}/{VAL_MAX}/{TEST_MAX} | alvo_pos={TARGET_POS_FRAC*100:.0f}%")
print(" - USE_PRETRAINED =", USE_PRETRAINED, "| FREEZE_ENCODER =", FREEZE_ENCODER, "| IMG_SIZE =", IMG_SIZE)


Fri Nov 28 03:01:18 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P0             49W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

# Segundo Ciclo - Aplicação de melhorias


In [ ]:
# -*- coding: utf-8 -*-
"""
CICLO 2 — Linha de base + melhorias:
- Pasta dedicada (cycle2/)
- Augmentations mais fortes (treino)
- Novas perdas: FOCAL e DICE+FOCAL
- Sweep de limiar mais fino
- Medição de tempo de treino por combinação
- Undersampling ATIVADO
- Figuras/Tabelas com prefixo do ciclo
"""

# =========================
# 0) HARDWARE & TF SETUP
# =========================
!nvidia-smi
!echo -n "RAM total: "; grep MemTotal /proc/meminfo

import os, sys, json, itertools, csv, random, functools, pickle, time
from typing import List, Tuple, Dict

import numpy as np
import pandas as pd
import tensorflow as tf

tf.config.optimizer.set_jit(False)
for g in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except Exception:
        pass

print("TF:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

# =========================
# 1) MODOS/FLAGS DO CICLO
# =========================
FAST_DEBUG = False
SWEEP_ONLY_05 = False

USE_PRETRAINED = True
FREEZE_ENCODER = False
EPOCHS = 20
BATCH_SIZE = 16
IMG_SIZE = (256, 256)
LR = 1e-4
PATIENCE = 10

# -------- CONTROLES DO CICLO 2 --------
USE_SUBSAMPLE            = False      # manter dataset completo
APPLY_UNDERSAMPLING      = True       # <<<< ATIVADO no CICLO 2
APPLY_TRAIN_AUGMENTATION = True       # <<<< ATIVADO no CICLO 2
# --------------------------------------

TRAIN_MAX = 4800
VAL_MAX   = 800
TEST_MAX  = 800
TARGET_POS_FRAC = 0.30
RANDOM_SEED = 1337

# Mixed precision estável (cabeça em float32)
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("float32")
print("Policy:", mixed_precision.global_policy())
tf.keras.backend.clear_session()

with tf.device("/GPU:0"):
    a = tf.random.normal([8000, 8000])
    _ = tf.matmul(a, a)
print("OK na GPU:", _ .shape)

# =========================
# 2) VIS & SALVAMENTO
# =========================
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams["savefig.dpi"]  = 600
mpl.rcParams["figure.dpi"]   = 120
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"]  = 42

from pathlib import Path
def savefig600(fig, basepath_no_ext: str):
    try:
        fig.tight_layout()
    except Exception:
        pass
    fig.savefig(str(basepath_no_ext) + ".pdf", bbox_inches="tight",
                metadata={"Creator":"ciclo2_treino.py"})
    fig.savefig(str(basepath_no_ext) + ".png", dpi=600, bbox_inches="tight",
                metadata={"Creator":"ciclo2_treino.py"})

# =========================
# 3) DRIVE & CAMINHOS
# =========================
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

# copia dados p/ local (caminho com acento: cuidado ao editar)
!rsync -ah --info=progress2 "/content/drive/MyDrive/[Mestrado] Repositório da Pesquisa/data/" "/content/data_local/"

BASE_DIR = "/content/data_local"
FALSE_COLOR_DIR        = os.path.join(BASE_DIR, "false_color")
MASKS_DIR              = os.path.join(BASE_DIR, "masks")
IMAGE_DISTRIBUTION_DIR = os.path.join(BASE_DIR, "image_distribution")

for p in [FALSE_COLOR_DIR, MASKS_DIR, IMAGE_DISTRIBUTION_DIR]:
    if not os.path.exists(p):
        print(f"[AVISO] Diretório inexistente: {p}")

# Pasta dedicada do CICLO 2
CYCLE_TAG = "cycle2"
OUT_DIR = Path(f"/content/drive/MyDrive/segm_resultados/{CYCLE_TAG}/reports_metrics")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("OUT_DIR:", OUT_DIR)

# =========================
# 4) LISTAS & PATHS
# =========================
def read_filelist(txt_path: str) -> List[str]:
    if not os.path.isfile(txt_path):
        raise FileNotFoundError(f"Arquivo de split não encontrado: {txt_path}")
    with open(txt_path, "r", encoding="utf-8") as f:
        names = [os.path.splitext(line.strip())[0] for line in f if line.strip()]
    return names

def make_full_paths(file_list: List[str], img_dir: str, mask_dir: str) -> Tuple[List[str], List[str]]:
    image_paths = [os.path.join(img_dir,  f + ".tif") for f in file_list]
    mask_paths  = [os.path.join(mask_dir, f + ".tif") for f in file_list]
    keep_img, keep_msk, miss = [], [], 0
    for ip, mp in zip(image_paths, mask_paths):
        if os.path.isfile(ip) and os.path.isfile(mp):
            keep_img.append(ip); keep_msk.append(mp)
        else:
            miss += 1
    if miss:
        print(f"[AVISO] {miss} pares ausentes (imagem/máscara não encontrados).")
    return keep_img, keep_msk

train_txt = os.path.join(IMAGE_DISTRIBUTION_DIR, "train.txt")
val_txt   = os.path.join(IMAGE_DISTRIBUTION_DIR, "val.txt")
test_txt  = os.path.join(IMAGE_DISTRIBUTION_DIR, "test.txt")

train_files = read_filelist(train_txt)
val_files   = read_filelist(val_txt)
train_image_paths, train_mask_paths = make_full_paths(train_files, FALSE_COLOR_DIR, MASKS_DIR)
val_image_paths,   val_mask_paths   = make_full_paths(val_files,   FALSE_COLOR_DIR, MASKS_DIR)

if os.path.isfile(test_txt):
    test_files = read_filelist(test_txt)
    test_image_paths, test_mask_paths = make_full_paths(test_files, FALSE_COLOR_DIR, MASKS_DIR)
    print(f"[INFO] test.txt encontrado: {len(test_image_paths)} amostras.")
else:
    test_image_paths, test_mask_paths = val_image_paths[:], val_mask_paths[:]
    print("[AVISO] test.txt não encontrado. Usando VAL também como TESTE.")

# =========================
# 5) SUBSAMPLE ESTRATIFICADO (desligado no ciclo 2)
# =========================
import imageio.v2 as imageio
random.seed(RANDOM_SEED)

def _has_fire(mask_path: str) -> bool:
    arr = imageio.imread(mask_path)
    if arr.ndim == 3:
        arr = arr[..., 0]
    arr = arr.astype(np.float32)
    thr = 0.5 * (arr.max() if arr.max() > 1 else 1.0)
    return bool((arr >= thr).any())

def stratified_subsample(img_paths, msk_paths, max_n, target_pos_frac=0.3):
    if (not USE_SUBSAMPLE) or (max_n is None) or (max_n <= 0) or (len(img_paths) <= max_n):
        return img_paths, msk_paths
    pos, neg = [], []
    for ip, mp in zip(img_paths, msk_paths):
        (pos if _has_fire(mp) else neg).append((ip, mp))
    n_pos_avail, n_neg_avail = len(pos), len(neg)
    n_pos_target = int(round(max_n * float(target_pos_frac)))
    n_neg_target = max_n - n_pos_target
    n_pos_take = min(n_pos_target, n_pos_avail)
    n_neg_take = min(n_neg_target + (n_pos_target - n_pos_take), n_neg_avail)
    while (n_pos_take + n_neg_take) < max_n:
        if n_neg_take < n_neg_avail: n_neg_take += 1
        elif n_pos_take < n_pos_avail: n_pos_take += 1
        else: break
    random.shuffle(pos); random.shuffle(neg)
    picked = pos[:n_pos_take] + neg[:n_neg_take]
    random.shuffle(picked)
    new_img = [p[0] for p in picked]
    new_msk = [p[1] for p in picked]
    print(f"[SUBSAMPLE] max_n={max_n} | pos_avail={n_pos_avail} neg_avail={n_neg_avail} "
          f"=> take_pos={n_pos_take} take_neg={n_neg_take} -> total={len(new_img)}")
    return new_img, new_msk

if USE_SUBSAMPLE:
    train_image_paths, train_mask_paths = stratified_subsample(
        train_image_paths, train_mask_paths, TRAIN_MAX, TARGET_POS_FRAC
    )
    val_image_paths, val_mask_paths = stratified_subsample(
        val_image_paths, val_mask_paths, VAL_MAX, TARGET_POS_FRAC
    )
    if 'test_image_paths' in locals() and len(test_image_paths) > 0:
        test_image_paths, test_mask_paths = stratified_subsample(
            test_image_paths, test_mask_paths, TEST_MAX, TARGET_POS_FRAC
        )

def count_fire_pairs(msk_paths):
    return sum(_has_fire(m) for m in msk_paths)

print(f"[CHECK] TREINO: {len(train_mask_paths)} amostras | com fogo: {count_fire_pairs(train_mask_paths)}")
print(f"[CHECK] VAL   : {len(val_mask_paths)} amostras | com fogo: {count_fire_pairs(val_mask_paths)}")
print(f"[CHECK] TESTE : {len(test_mask_paths)} amostras | com fogo: {count_fire_pairs(test_mask_paths)}")

# =========================
# 6) BALANCEAMENTO (UNDERSAMPLING — ATIVADO)
# =========================
from PIL import Image
from tqdm import tqdm

def balancear_dataset_por_undersampling(image_paths: List[str], mask_paths: List[str]):
    positivos_img, positivos_mask, negativos_img, negativos_mask = [], [], [], []
    print("[INFO] Analisando máscaras para balanceamento (undersampling)...")
    for img_p, mask_p in tqdm(list(zip(image_paths, mask_paths)), total=len(image_paths)):
        if not os.path.exists(mask_p):
            continue
        with Image.open(mask_p) as mask:
            arr = np.array(mask)
            if np.sum(arr) > 0:
                positivos_img.append(img_p); positivos_mask.append(mask_p)
            else:
                negativos_img.append(img_p); negativos_mask.append(mask_p)
    num_pos, num_neg = len(positivos_img), len(negativos_img)
    print(f"[INFO] Positivas: {num_pos} | Negativas: {num_neg}")
    if num_pos == 0 or num_neg == 0:
        print("[WARN] Desbalanceado extremo. Retornando original.")
        combined = list(zip(image_paths, mask_paths)); random.shuffle(combined)
        new_image_paths, new_mask_paths = zip(*combined)
        return list(new_image_paths), list(new_mask_paths)
    # reduzir a classe MAIORIA para o tamanho da minoria
    if num_pos > num_neg:
        reduzir = num_neg
        pos_sample = random.sample(list(zip(positivos_img, positivos_mask)), reduzir)
        final_pos_img, final_pos_mask = zip(*pos_sample)
        final_neg_img, final_neg_mask = negativos_img, negativos_mask
    else:
        reduzir = num_pos
        neg_sample = random.sample(list(zip(negativos_img, negativos_mask)), reduzir)
        final_neg_img, final_neg_mask = zip(*neg_sample)
        final_pos_img, final_pos_mask = positivos_img, positivos_mask
    new_image_paths = list(final_pos_img) + list(final_neg_img)
    new_mask_paths  = list(final_pos_mask) + list(final_neg_mask)
    combined = list(zip(new_image_paths, new_mask_paths)); random.shuffle(combined)
    new_image_paths, new_mask_paths = zip(*combined)
    print(f"[INFO] Balanceamento concluído. Total final: {len(new_image_paths)}")
    return list(new_image_paths), list(new_mask_paths)

if APPLY_UNDERSAMPLING:
    train_image_paths_bal, train_mask_paths_bal = balancear_dataset_por_undersampling(
        train_image_paths, train_mask_paths
    )
else:
    print("[INFO] Undersampling desativado: usando dataset de treino ORIGINAL.")
    train_image_paths_bal, train_mask_paths_bal = train_image_paths, train_mask_paths

# =========================
# 7) AUGMENTATIONS
# =========================
import albumentations as A

def get_train_augmentations(img_size: Tuple[int, int]):
    # pacote de augmentations do CICLO 2 (moderado-forte)
    return A.Compose([
        A.Resize(height=img_size[0], width=img_size[1]),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.10, rotate_limit=15,
                           border_mode=0, p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=10, val_shift_limit=10, p=0.3),
        A.GaussianBlur(blur_limit=(3,5), p=0.2),
    ])

def get_val_augmentations(img_size: Tuple[int, int]):
    return A.Compose([A.Resize(height=img_size[0], width=img_size[1])])

if APPLY_TRAIN_AUGMENTATION:
    train_augs = get_train_augmentations(IMG_SIZE)
else:
    print("[INFO] Data augmentation no TREINO desativada: apenas Resize.")
    train_augs = get_val_augmentations(IMG_SIZE)

val_augs = get_val_augmentations(IMG_SIZE)

# =========================
# 8) LOADER .tif + tf.data
# =========================
import imageio.v2 as imageio

def _read_tiff_rgb(img_path: str) -> np.ndarray:
    arr = imageio.imread(img_path)
    if arr.ndim == 2:
        arr = np.stack([arr, arr, arr], axis=-1)
    elif arr.ndim == 3:
        if arr.shape[-1] == 4:   arr = arr[..., :3]
        elif arr.shape[-1] > 3:  arr = arr[..., :3]
        elif arr.shape[-1] == 1: arr = np.repeat(arr, 3, axis=-1)
    arr = arr.astype(np.float32)
    maxv = arr.max() if arr.size else 1.0
    if maxv > 1.0:
        arr = arr / (255.0 if maxv <= 255 else maxv)
    return arr

def _read_tiff_mask(msk_path: str) -> np.ndarray:
    arr = imageio.imread(msk_path)
    if arr.ndim == 3:
        arr = arr[..., 0]
    arr = arr.astype(np.float32)
    thr = 0.5 * (arr.max() if arr.max() > 1 else 1.0)
    binm = (arr >= thr).astype(np.float32)
    binm = np.expand_dims(binm, axis=-1)
    return binm

def _apply_augs_np(img, msk, augs):
    if augs is None:
        return img, msk
    try:
        out = augs(image=img, mask=msk.squeeze(-1))
        img2, msk2 = out["image"], out["mask"]
        if msk2.ndim == 2:
            msk2 = np.expand_dims(msk2, axis=-1)
        msk2 = (msk2 >= 0.5).astype(np.float32)
        return img2.astype(np.float32), msk2.astype(np.float32)
    except Exception:
        return img, msk

def _resize_np(img: np.ndarray, msk: np.ndarray, size=IMG_SIZE):
    try:
        import cv2
        img_r = cv2.resize(img, dsize=(size[1], size[0]), interpolation=cv2.INTER_LINEAR)
        msk_r = cv2.resize(msk, dsize=(size[1], size[0]), interpolation=cv2.INTER_NEAREST)
    except Exception:
        img_r = tf.image.resize(img, size, method="bilinear").numpy()
        msk_r = tf.image.resize(msk, size, method="nearest").numpy()
    if msk_r.ndim == 2:
        msk_r = msk_r[..., None]
    return img_r.astype(np.float32), msk_r.astype(np.float32)

def load_pair_np(img_path_b, msk_path_b, size=IMG_SIZE, augs=None):
    ip = img_path_b.decode("utf-8"); mp = msk_path_b.decode("utf-8")
    img = _read_tiff_rgb(ip)
    msk = _read_tiff_mask(mp)
    img, msk = _resize_np(img, msk, size=size)
    img, msk = _apply_augs_np(img, msk, augs)
    if msk.ndim == 2: msk = msk[..., None]
    msk = (msk >= 0.5).astype(np.float32)
    img = np.clip(img, 0.0, 1.0).astype(np.float32)
    return img, msk

def _py_load(img_p, msk_p, size, augs):
    img, msk = tf.numpy_function(
        func=lambda ip, mp: load_pair_np(ip, mp, size=size, augs=augs),
        inp=[img_p, msk_p],
        Tout=[tf.float32, tf.float32]
    )
    img.set_shape((IMG_SIZE[0], IMG_SIZE[1], 3))
    msk.set_shape((IMG_SIZE[0], IMG_SIZE[1], 1))
    return img, msk

def make_tfds(image_paths: List[str], mask_paths: List[str], batch_size: int,
              shuffle: bool, augs=None, cache=True):
    ds = tf.data.Dataset.from_tensor_slices((image_paths, mask_paths))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(image_paths), 2000), reshuffle_each_iteration=True)
    ds = ds.map(lambda ip, mp: _py_load(ip, mp, IMG_SIZE, augs), num_parallel_calls=tf.data.AUTOTUNE)
    if cache:
        ds = ds.cache()
    ds = ds.batch(batch_size, drop_remainder=False).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_tfds(train_image_paths_bal, train_mask_paths_bal, BATCH_SIZE, True,  augs=train_augs, cache=True)
val_ds   = make_tfds(val_image_paths,       val_mask_paths,       BATCH_SIZE, False, augs=val_augs,   cache=True)
test_ds  = make_tfds(test_image_paths,      test_mask_paths,      BATCH_SIZE, False, augs=None,       cache=True)

for xb, yb in train_ds.take(1):
    print("[OK] Batch exemplo:", xb.shape, yb.shape)

# =========================
# 9) MÉTRICAS STREAMING
# =========================
class StreamingConfusion(tf.keras.metrics.Metric):
    def __init__(self, threshold=0.5, name="streaming_conf", **kwargs):
        super().__init__(name=name, **kwargs)
        self.threshold = threshold
        self.tp = self.add_weight(name="tp", shape=(), initializer="zeros", dtype=tf.float32)
        self.fp = self.add_weight(name="fp", shape=(), initializer="zeros", dtype=tf.float32)
        self.fn = self.add_weight(name="fn", shape=(), initializer="zeros", dtype=tf.float32)
        self.tn = self.add_weight(name="tn", shape=(), initializer="zeros", dtype=tf.float32)
    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = tf.cast(y_true > 0.5, tf.float32)
        y_pred = tf.cast(y_pred >= self.threshold, tf.float32)
        tp = tf.reduce_sum(y_true * y_pred)
        fp = tf.reduce_sum((1.0 - y_true) * y_pred)
        fn = tf.reduce_sum(y_true * (1.0 - y_pred))
        tn = tf.reduce_sum((1.0 - y_true) * (1.0 - y_pred))
        self.tp.assign_add(tp); self.fp.assign_add(fp)
        self.fn.assign_add(fn); self.tn.assign_add(tn)
    def reset_state(self):
        for v in (self.tp, self.fp, self.fn, self.tn):
            v.assign(0.0)

class StreamingPrecision(StreamingConfusion):
    def result(self):
        denom = self.tp + self.fp
        return tf.where(denom > 0, self.tp / denom, 0.0)

class StreamingRecall(StreamingConfusion):
    def result(self):
        denom = self.tp + self.fn
        return tf.where(denom > 0, self.tp / denom, 0.0)

class StreamingF1(StreamingConfusion):
    def result(self):
        num = 2.0 * self.tp
        denom = 2.0 * self.tp + self.fp + self.fn
        return tf.where(denom > 0, num / denom, 0.0)

class StreamingIoU(StreamingConfusion):
    def result(self):
        denom = self.tp + self.fp + self.fn
        return tf.where(denom > 0, self.tp / denom, 0.0)

# =========================
# 10) LOSSES & COMPILAÇÃO
# =========================
def dice_loss(y_true, y_pred, smooth: float = 1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1,2,3])
    denom = tf.reduce_sum(y_true + y_pred, axis=[1,2,3])
    dice = (2.0 * intersection + smooth) / (denom + smooth)
    return 1.0 - dice

BCE_LOSS = tf.keras.losses.BinaryCrossentropy(from_logits=False)

def focal_loss(gamma: float = 2.0, alpha: float = 0.25):
    def _loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(tf.cast(y_pred, tf.float32), 1e-7, 1.0 - 1e-7)
        ce = -(y_true * tf.math.log(y_pred) + (1.0 - y_true) * tf.math.log(1.0 - y_pred))
        p_t = y_true * y_pred + (1.0 - y_true) * (1.0 - y_pred)
        w   = alpha * tf.pow(1.0 - p_t, gamma)
        return tf.reduce_mean(w * ce)
    return _loss

LOSS_FUNCS = {
    "BCE"       : BCE_LOSS,
    "DICE"      : dice_loss,
    "FOCAL"     : focal_loss(gamma=2.0, alpha=0.25),
    "DICEFOCAL" : lambda y_true, y_pred: 0.5*dice_loss(y_true, y_pred) +
                                         0.5*focal_loss(2.0, 0.25)(y_true, y_pred),
}

def compile_with_streaming(model, loss_fn, threshold=0.5, lr=LR):
    opt = tf.keras.optimizers.Adam(learning_rate=lr)
    metrics = [
        StreamingIoU(threshold=threshold, name="iou"),
        StreamingF1 (threshold=threshold, name="f1"),
        StreamingPrecision(threshold=threshold, name="precision"),
        StreamingRecall  (threshold=threshold, name="recall"),
    ]
    model.compile(optimizer=opt, loss=loss_fn, metrics=metrics)
    return model

# =========================
# 11) MODELO (U-NET)
# =========================
from tensorflow.keras import layers as L
from tensorflow.keras.applications import VGG16, EfficientNetB3

def _bn32(x): return L.BatchNormalization(dtype="float32")(x)

def conv_block(x, filters):
    x = L.Conv2D(filters, 3, padding="same")(x); x = _bn32(x); x = L.ReLU()(x)
    x = L.Conv2D(filters, 3, padding="same")(x); x = _bn32(x); x = L.ReLU()(x)
    return x

def up_block(x, skip, filters):
    x = L.UpSampling2D((2,2), interpolation="bilinear")(x)
    x = tf.keras.layers.Resizing(skip.shape[1], skip.shape[2], interpolation="bilinear")(x)
    x = L.Concatenate()([x, skip])
    x = conv_block(x, filters)
    return x

def _check_downsample_ratio(inp_shape_hw, feat_shape_hw, expected_ratio):
    H_in, W_in = int(inp_shape_hw[0]), int(inp_shape_hw[1])
    H_f,  W_f  = int(feat_shape_hw[0]), int(feat_shape_hw[1])
    assert H_f > 0 and W_f > 0
    assert H_in % H_f == 0 and W_in % W_f == 0
    rh, rw = H_in // H_f, W_in // W_f
    assert rh == expected_ratio and rw == expected_ratio, (
        f"Esperado downsample {expected_ratio}×, obtido {rh}×{rw}."
    )

def _vgg16_endpoints(base, inputs_shape):
    s1 = base.get_layer("block1_conv2").output   # 1x
    s2 = base.get_layer("block2_conv2").output   # 2x
    s3 = base.get_layer("block3_conv3").output   # 4x
    s4 = base.get_layer("block4_conv3").output   # 8x
    b  = base.get_layer("block5_conv3").output   # 16x
    _check_downsample_ratio(inputs_shape[:2], s1.shape[1:3], 1)
    _check_downsample_ratio(inputs_shape[:2], s2.shape[1:3], 2)
    _check_downsample_ratio(inputs_shape[:2], s3.shape[1:3], 4)
    _check_downsample_ratio(inputs_shape[:2], s4.shape[1:3], 8)
    _check_downsample_ratio(inputs_shape[:2], b.shape[1:3], 16)
    return s1, s2, s3, s4, b

def _efficientnetb3_endpoints(base, inputs_shape):
    s0 = base.get_layer("block2b_add").output    # /4
    s1 = base.get_layer("block3b_add").output    # /8
    s2 = base.get_layer("block4c_add").output    # /16
    s3 = base.get_layer("block5c_add").output    # /16
    b  = base.get_layer("block6d_add").output    # /32
    return s0, s1, s2, s3, b

def build_unet(backbone_name="vgg16", input_shape=(256,256,3), encoder_trainable=False):
    inputs = L.Input(shape=input_shape)
    if backbone_name.lower() == "vgg16":
        base = VGG16(include_top=False, weights="imagenet", input_tensor=inputs)
        for l in base.layers: l.trainable = encoder_trainable
        s1, s2, s3, s4, b = _vgg16_endpoints(base, input_shape)
        x = up_block(b,  s4, 512)
        x = up_block(x, s3, 256)
        x = up_block(x, s2, 128)
        x = up_block(x, s1, 64)
    elif backbone_name.lower() == "efficientnetb3":
        base = EfficientNetB3(include_top=False, weights="imagenet", input_tensor=inputs)
        for l in base.layers: l.trainable = encoder_trainable
        s0, s1, s2, s3, b = _efficientnetb3_endpoints(base, input_shape)
        x = up_block(b,  s3, 512)   # 1/16 -> 1/16
        x = up_block(x, s2, 256)    # 1/16 -> 1/8
        x = up_block(x, s1, 128)    # 1/8  -> 1/4
        x = up_block(x, s0, 64)     # 1/4  -> 1/2
        x = tf.keras.layers.Resizing(input_shape[0], input_shape[1], interpolation="bilinear")(x)
    else:
        raise ValueError("backbone_name deve ser 'vgg16' ou 'efficientnetb3'.")
    outputs = L.Conv2D(1, 1, activation="sigmoid", dtype="float32")(x)
    model = tf.keras.Model(inputs, outputs, name=f"UNet_{backbone_name}")
    print("\n[BACKBONE]", backbone_name.upper())
    print("Entrada :", input_shape)
    return model

# =========================
# 12) TREINO + PLOTS
# =========================
from matplotlib.ticker import MaxNLocator, AutoMinorLocator

def _style_axes_epoch(ax, epochs_total=None):
    ax.set_axisbelow(True)
    ax.grid(True, which="major", linestyle="--", linewidth=0.8, alpha=0.35)
    ax.grid(True, which="minor", linestyle=":",  linewidth=0.6, alpha=0.25)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.xaxis.set_minor_locator(AutoMinorLocator(2))
    if epochs_total is not None and epochs_total > 0:
        ax.set_xlim(0, max(1, int(epochs_total)))
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Metric Value")
    leg = ax.legend(frameon=True); leg.get_frame().set_alpha(0.8)

def plot_history(history, title, fname_base):
    # Loss
    fig, ax = plt.subplots(figsize=(7,4), dpi=150)
    ax.plot(history.history.get("loss", []),     label="Loss (train)",   linewidth=1.5)
    ax.plot(history.history.get("val_loss", []), label="Loss (val)",     linewidth=1.5)
    ax.set_title(f"{title} — Loss")
    _style_axes_epoch(ax, epochs_total=len(history.history.get("loss", [])))
    savefig600(fig, OUT_DIR / f"{CYCLE_TAG}_{fname_base}_loss")
    plt.close(fig)

    # Métricas
    fig, ax = plt.subplots(figsize=(7,4), dpi=150)
    if "precision" in history.history:     ax.plot(history.history["precision"],   label="Precision",   linewidth=1.5)
    if "recall" in history.history:        ax.plot(history.history["recall"],      label="Recall",      linewidth=1.5)
    if "iou" in history.history:           ax.plot(history.history["iou"],         label="IoU",         linewidth=1.5)
    if "f1" in history.history:            ax.plot(history.history["f1"],          label="F1 Score",    linewidth=1.5)
    if "val_iou" in history.history:       ax.plot(history.history["val_iou"],     label="Val IoU",     linewidth=1.5)
    if "val_f1" in history.history:        ax.plot(history.history["val_f1"],      label="Val F1",      linewidth=1.5)
    if "val_precision" in history.history: ax.plot(history.history["val_precision"], label="Val Precision", linewidth=1.5)
    if "val_recall" in history.history:    ax.plot(history.history["val_recall"],    label="Val Recall",    linewidth=1.5)
    ax.set_title(f"{title} — Metrics Over Epochs")
    ax.set_ylim(0.0, 1.0)
    _style_axes_epoch(ax, epochs_total=len(history.history.get("loss", [])))
    savefig600(fig, OUT_DIR / f"{CYCLE_TAG}_{fname_base}_metrics")
    plt.close(fig)

def fit_model(model, train_ds, val_ds, tag):
    ckpt_path = OUT_DIR / f"{CYCLE_TAG}_best_{tag}.weights.h5"
    cbs = [
        tf.keras.callbacks.EarlyStopping(monitor="val_f1", mode="max",
                                         patience=PATIENCE, restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint(filepath=str(ckpt_path),
                                           monitor="val_f1", mode="max",
                                           save_best_only=True, save_weights_only=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_f1", mode="max",
                                             factor=0.5, patience=max(3, PATIENCE//2), verbose=1),
    ]
    t0 = time.time()
    hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
                     callbacks=cbs, verbose=1)
    train_time = time.time() - t0
    (OUT_DIR / f"{CYCLE_TAG}_times.csv").open("a").write(f"{tag},{train_time:.1f}\n")
    return hist, ckpt_path

# =========================
# 13) SWEEP, AVAL, CM, TABELA
# =========================
def sweep_threshold(model, val_ds, thresholds=None):
    if thresholds is None:
        thresholds = np.linspace(0.10, 0.90, 41)   # passo 0.02
    best_t, best_f1 = thresholds[0], -1.0
    rows = []
    for t in thresholds:
        TP=FP=FN=TN=0
        for x, y in val_ds:
            y_true = (y.numpy() > 0.5).astype(np.uint8)
            y_prob = model.predict(x, verbose=0)
            y_pred = (y_prob >= t).astype(np.uint8)
            TP += (y_true & y_pred).sum()
            FP += ((1 - y_true) & y_pred).sum()
            FN += (y_true & (1 - y_pred)).sum()
            TN += ((1 - y_true) & (1 - y_pred)).sum()
        precision = TP/(TP+FP) if (TP+FP)>0 else 0.0
        recall    = TP/(TP+FN) if (TP+FN)>0 else 0.0
        f1        = (2*precision*recall)/(precision+recall) if (precision+recall)>0 else 0.0
        iou       = TP/(TP+FP+FN) if (TP+FP+FN)>0 else 0.0
        rows.append(dict(threshold=t, precision=precision, recall=recall, f1=f1, iou=iou,
                         TP=int(TP), FP=int(FP), FN=int(FN), TN=int(TN)))
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, pd.DataFrame(rows)

def evaluate_dataset(model, dataset, threshold=0.5):
    TP=FP=FN=TN=0
    for x, y in dataset:
        y_true = (y.numpy() > 0.5).astype(np.uint8)
        y_prob = model.predict(x, verbose=0)
        y_pred = (y_prob >= threshold).astype(np.uint8)
        TP += (y_true & y_pred).sum()
        FP += ((1 - y_true) & y_pred).sum()
        FN += (y_true & (1 - y_pred)).sum()
        TN += ((1 - y_true) & (1 - y_pred)).sum()
    precision = TP/(TP+FP) if (TP+FP)>0 else 0.0
    recall    = TP/(TP+FN) if (TP+FN)>0 else 0.0
    f1        = (2*precision*recall)/(precision+recall) if (precision+recall)>0 else 0.0
    iou       = TP/(TP+FP+FN) if (TP+FP+FN)>0 else 0.0
    cm = np.array([[TN, FP],[FN, TP]], dtype=np.int64)
    return dict(precision=precision, recall=recall, f1=f1, iou=iou,
                TP=int(TP), FP=int(FP), FN=int(FN), TN=int(TN), threshold=threshold), cm

def plot_confusion_matrix(
    cm, classes=("0","1"), title="Confusion Matrix",
    normalize=None, png_path=None, pdf_path=None,
):
    cm = np.asarray(cm)
    show_norm = normalize in ("true", "all")
    cm_plot = cm.astype(np.float64).copy()
    if normalize == "true":
        row_sums = cm_plot.sum(axis=1, keepdims=True)
        cm_plot = np.divide(cm_plot, row_sums, out=np.zeros_like(cm_plot), where=row_sums > 0.0)
        vmin, vmax = 0.0, 1.0
    elif normalize == "all":
        tot = cm_plot.sum()
        cm_plot = cm_plot / tot if tot > 0 else cm_plot
        vmin, vmax = 0.0, 1.0
    else:
        vmin, vmax = 0, cm_plot.max() if cm_plot.size else 1
    fig, ax = plt.subplots(figsize=(6, 6), dpi=150)
    im = ax.imshow(cm_plot, cmap="Greens", vmin=vmin, vmax=vmax, interpolation="nearest")
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.set_ylabel("Proporção" if show_norm else "Contagem", rotation=270, labelpad=12)
    ax.set_title(title, pad=10)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_xticks(np.arange(len(classes)), labels=classes)
    ax.set_yticks(np.arange(len(classes)), labels=classes)
    ax.set_aspect("equal")
    thresh = (cm_plot.max() / 2.0) if cm_plot.size else 0.0
    for i in range(cm_plot.shape[0]):
        for j in range(cm_plot.shape[1]):
            if show_norm: text = f"{cm_plot[i, j]:.2f}\n({int(cm[i, j])})"
            else:         text = f"{int(cm_plot[i, j])}"
            ax.text(j, i, text, ha="center", va="center",
                    fontsize=12, color="white" if cm_plot[i, j] > thresh else "black")
    plt.tight_layout()
    if pdf_path: fig.savefig(pdf_path, bbox_inches="tight")
    if png_path: fig.savefig(png_path, dpi=600, bbox_inches="tight")
    plt.close(fig)

def save_table2(val_metrics, test_metrics, backbone_tag_loss, out_dir=OUT_DIR):
    # backbone_tag_loss ex.: "vgg16_BCE"
    backbone_tag, loss_name = backbone_tag_loss.split("_", 1)
    df = pd.DataFrame([
        {"Modelo": f"{backbone_tag.upper()} ({loss_name})", "Conjunto":"Validação",
         "Threshold": f"{val_metrics['threshold']:.2f}",
         "Precisão": val_metrics["precision"],
         "Recall":   val_metrics["recall"],
         "F1-Score": val_metrics["f1"],
         "IoU":      val_metrics["iou"]},
        {"Modelo": f"{backbone_tag.upper()} ({loss_name})", "Conjunto":"Teste",
         "Threshold": f"{test_metrics['threshold']:.2f}",
         "Precisão": test_metrics["precision"],
         "Recall":   test_metrics["recall"],
         "F1-Score": test_metrics["f1"],
         "IoU":      test_metrics["iou"]},
    ])
    csv_path = out_dir / f"{CYCLE_TAG}_tabela2_{backbone_tag}_{loss_name}.csv"
    tex_path = out_dir / f"{CYCLE_TAG}_tabela2_{backbone_tag}_{loss_name}.tex"
    df.to_csv(csv_path, index=False)
    with open(tex_path, "w", encoding="utf-8") as f:
        f.write(df.to_latex(index=False, float_format="%.4f", escape=False,
                caption=(f"Tabela 2 — {backbone_tag.upper()} (loss: {loss_name}) "
                         "com métricas por matrizes de confusão (Validação/Teste) no limiar ótimo."),
                label=f"tab:{CYCLE_TAG}_metrics_{backbone_tag}_{loss_name}"))
    print(f"[OK] {backbone_tag}_{loss_name}: Tabela 2 salva em {csv_path} e {tex_path}")
    return df

def run_pipeline_for(backbone_tag, loss_name, train_ds, val_ds, test_ds):
    print(f"\n========== {backbone_tag.upper()} ({loss_name}) ==========")
    model = build_unet(backbone_tag, input_shape=(*IMG_SIZE, 3))
    model = compile_with_streaming(model, loss_fn=LOSS_FUNCS[loss_name], threshold=0.5, lr=LR)

    tag = f"{backbone_tag}_{loss_name}"
    history, ckpt_path = fit_model(model, train_ds, val_ds, tag)

    fig_id = "8" if backbone_tag.lower()=="vgg16" else "9"
    plot_history(history, f"U-Net ({backbone_tag.upper()} / {loss_name})",
                 fname_base=f"Figura_{fig_id}_{backbone_tag}_{loss_name}_curvas")

    model.load_weights(str(ckpt_path))

    best_t, sweep_df = sweep_threshold(model, val_ds)
    json_path = OUT_DIR / f"{CYCLE_TAG}_best_thresholds.json"
    try:
        data = json.loads(json_path.read_text())
    except Exception:
        data = {}
    if loss_name not in data: data[loss_name] = {}
    data[loss_name][backbone_tag] = float(best_t)
    json_path.write_text(json.dumps(data, indent=2))

    sweep_df.to_csv(OUT_DIR / f"{CYCLE_TAG}_sweep_{backbone_tag}_{loss_name}.csv", index=False)
    print(f"[{backbone_tag}/{loss_name}] Threshold ótimo (val/F1): {best_t:.2f}")

    val_metrics,  val_cm  = evaluate_dataset(model, val_ds,  threshold=best_t)
    test_metrics, test_cm = evaluate_dataset(model, test_ds, threshold=best_t)
    print(f"[{backbone_tag}/{loss_name}] VAL:", val_metrics)
    print(f"[{backbone_tag}/{loss_name}] TEST:", test_metrics)

    # Figuras (Val/Test) — absoluta e normalizada
    plot_confusion_matrix(val_cm,  title=f"{backbone_tag.upper()} — Validação (thr={best_t:.2f})",
                          normalize=None,
                          png_path=OUT_DIR / f"{CYCLE_TAG}_Figura_10_{backbone_tag}_{loss_name}_val_cm.png",
                          pdf_path=OUT_DIR / f"{CYCLE_TAG}_Figura_10_{backbone_tag}_{loss_name}_val_cm.pdf")
    plot_confusion_matrix(val_cm,  title=f"{backbone_tag.upper()} — Validação Normalizada (thr={best_t:.2f})",
                          normalize='true',
                          png_path=OUT_DIR / f"{CYCLE_TAG}_Figura_10_{backbone_tag}_{loss_name}_val_cm_norm.png",
                          pdf_path=OUT_DIR / f"{CYCLE_TAG}_Figura_10_{backbone_tag}_{loss_name}_val_cm_norm.pdf")
    plot_confusion_matrix(test_cm, title=f"{backbone_tag.upper()} — Teste (thr={best_t:.2f})",
                          normalize=None,
                          png_path=OUT_DIR / f"{CYCLE_TAG}_Figura_11_{backbone_tag}_{loss_name}_test_cm.png",
                          pdf_path=OUT_DIR / f"{CYCLE_TAG}_Figura_11_{backbone_tag}_{loss_name}_test_cm.pdf")
    plot_confusion_matrix(test_cm, title=f"{backbone_tag.upper()} — Teste Normalizada (thr={best_t:.2f})",
                          normalize='true',
                          png_path=OUT_DIR / f"{CYCLE_TAG}_Figura_11_{backbone_tag}_{loss_name}_test_cm_norm.png",
                          pdf_path=OUT_DIR / f"{CYCLE_TAG}_Figura_11_{backbone_tag}_{loss_name}_test_cm_norm.pdf")

    _ = save_table2(val_metrics, test_metrics, f"{backbone_tag}_{loss_name}", OUT_DIR)
    return dict(backbone=backbone_tag, loss=loss_name,
                best_threshold=best_t, val=val_metrics, test=test_metrics)

# =========================
# 14) EXECUTAR
# =========================
BACKBONES = ["vgg16", "efficientnetb3"]
LOSSES_TO_RUN = ["BCE", "DICE", "FOCAL", "DICEFOCAL"]  # 4 perdas no CICLO 2

results = []
for loss_name in LOSSES_TO_RUN:
    for bb in BACKBONES:
        results.append(run_pipeline_for(bb, loss_name, train_ds, val_ds, test_ds))

# Tabela comparativa geral (Figura 2)
rows = []
for r in results:
    rows += [
        {"Backbone": r["backbone"].upper(), "Loss": r["loss"], "Conjunto":"Validação",
         "Threshold": f"{r['best_threshold']:.2f}",
         "Precisão": r["val"]["precision"], "Recall": r["val"]["recall"],
         "F1-Score": r["val"]["f1"], "IoU": r["val"]["iou"]},
        {"Backbone": r["backbone"].upper(), "Loss": r["loss"], "Conjunto":"Teste",
         "Threshold": f"{r['best_threshold']:.2f}",
         "Precisão": r["test"]["precision"], "Recall": r["test"]["recall"],
         "F1-Score": r["test"]["f1"], "IoU": r["test"]["iou"]},
    ]

df_all = pd.DataFrame(rows).sort_values(["Backbone","Loss","Conjunto"])
df_all.to_csv(OUT_DIR / f"{CYCLE_TAG}_Figura_2_tabela2_comparativo.csv", index=False)
with open(OUT_DIR / f"{CYCLE_TAG}_Figura_2_tabela2_comparativo.tex", "w", encoding="utf-8") as f:
    f.write(df_all.to_latex(index=False, float_format="%.4f", escape=False,
            caption=(f"Tabela 2 — {CYCLE_TAG.upper()} — Comparativo (Backbones × Funções de Perda) "
                     "com métricas pixel-a-pixel calculadas a partir das matrizes de confusão, "
                     "no limiar ótimo de validação."),
            label=f"tab:{CYCLE_TAG}_tabela2_comparativo"))

print("\n[PRONTO] Artefatos do CICLO 2 em:", OUT_DIR.resolve())
print(" - Figuras (curvas e CMs) em PDF + PNG 600dpi")
print(" - Tabelas .csv e .tex (Val/Teste + threshold) por backbone+loss")
print(f" - {CYCLE_TAG}_Figura_2_tabela2_comparativo.(csv|tex)")
print(f" - Undersampling = {APPLY_UNDERSAMPLING} | Augmentations Treino = {APPLY_TRAIN_AUGMENTATION}")


Thu Dec  4 12:48:19 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   29C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

100%|██████████| 5100/5100 [00:03<00:00, 1618.57it/s]


[INFO] Positivas: 2650 | Negativas: 2450
[INFO] Balanceamento concluído. Total final: 4900


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


[OK] Batch exemplo: (16, 256, 256, 3) (16, 256, 256, 1)

========== VGG16 (BCE) ==========
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

[BACKBONE] VGG16
Entrada : (256, 256, 3)
Epoch 1/20
307/307 ━━━━━━━━━━━━━━━━━━━━ 0s 211ms/step - f1: 0.5567 - iou: 0.3976 - loss: 0.1175 - precision: 0.4492 - recall: 0.7833
Epoch 1: val_f1 improved from -inf to 0.85662, saving model to /content/drive/MyDrive/segm_resultados/cycle2/reports_metrics/cycle2_best_vgg16_BCE.weights.h5
307/307 ━━━━━━━━━━━━━━━━━━━━ 120s 268ms/step - f1: 0.5571 - iou: 0.3980 - loss: 0.1173 - precision: 0.4498 - recall: 0.7833 - val_f1: 0.8566 - val_iou: 0.7492 - val_loss: 0.0481 - val_precision: 0.7664 - val_recall: 0.9710 - learning_rate: 1.0000e-04
Epoch 2/20
306/307 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step - f1: 0.8513 - iou: 0.7411 - loss: 0.0306 - precision: 0.8690 - recall: 0.8343
Epoch 2: val_f1 improved from 0.85662 to 0.91846, saving model to /content/drive/MyDrive/segm_resultados/cycle2/reports_metrics/cycle2_best_v